In [9]:
import pandas as pd
from io import StringIO  # For testing with string data

# Sample data (you would typically read this from a file)
with open('out_details.ts1', 'r') as file:
    data = file.readlines()


# Simulate reading from a file
# You can use `open('data.txt', 'r')` in your actual implementation
lines = data.strip().split('\n')

# Initialize a list to store the relevant data
table_data = []
recording = False  # Flag to start recording data

# Read lines to find the relevant table
for line in lines:
    # Check for the start of the relevant data section
    if line.startswith('Nr'):
        recording = True  # Start recording data
        table_data.append(line.strip())  # Include header line
        continue
    # Check for the end of the relevant data section
    if recording and line.strip() == "":
        break  # Stop recording on a blank line
    # If we're in the data section, add the line
    if recording:
        table_data.append(line.strip())

# Load the relevant data into a pandas DataFrame
df = pd.read_csv(StringIO('\n'.join(table_data)), delim_whitespace=True)

# Display the DataFrame
print("Extracted Table Data:")
print(df)

# Optionally filter for J_z = 0.5 and find the maximum energy
filtered_df = df[df['j_z'] == 0.5]
if not filtered_df.empty:
    max_energy = filtered_df['energy'].max()
    print(f"The maximum energy corresponding to J_z = 0.5 is: {max_energy}")
else:
    print("No data for J_z = 0.5 found.")


AttributeError: 'list' object has no attribute 'strip'

In [21]:
import pandas as pd
import re

# File path (replace with your actual file path)
file_path = 'out_details.ts1'

# Read the entire file into memory
with open(file_path, 'r') as file:
    content = file.read()

# Use regex to find all relevant tables
# The regex assumes that each table starts with 'Nr' and ends with a blank line
tables = re.findall(r'Nr.*?(?=\n\s*\n)', content, re.DOTALL)

# Initialize a list to hold DataFrames for each table
dataframes = []

# Process each table
for table in tables:
    # Split the table into lines and clean up
    lines = table.strip().split('\n')
    
    # Load the relevant data into a pandas DataFrame
    try:
        df = pd.read_csv(StringIO('\n'.join(lines)), delim_whitespace=True)
        dataframes.append(df)
    except pd.errors.EmptyDataError:
        print("Empty DataFrame encountered for a table. Skipping...")

# Display extracted DataFrames and find maximum energy for J_z = 0.5 in each
for i, df in enumerate(dataframes):
    print(f"Table {i + 1}:")
    is_numeric = pd.to_numeric(df.iloc[-1], errors='coerce').notnull().all()
    if is_numeric == False:
        df.drop(df.tail(1).index,inplace=True) # drop last n rows
    # Filter for J_z = 0.5 and find the maximum energy
    if 'energy' in df.columns:
        df['Iso'] = df['Iso'].astype(float)
        filtered_df = df[df['Iso'] == 0.5]
        print(filtered_df)
        if not filtered_df.empty:
            max_energy = filtered_df['delta'].max()
            print(f"The maximum energy corresponding to J_z = 0.5 in Table {i + 1} is: {max_energy}\n")
        else:
            print(f"No data for J_z = 0.5 found in Table {i + 1}.\n")


Table 1:
Table 2:
   Nr  Iso j_z tot    r        z   wtpair  v^2    uv   delta    energy  \
1   1  0.5   1   1  1.0  1.00000  1.00000  0.0 -0.01 -39.836  0.000810   
8   1  0.5  -1   1  1.0  1.00000  1.00000  0.0 -0.01 -33.608  0.014091   
2   1  0.5  -1   1  1.0  1.00000  1.00000  0.0 -0.01 -32.619  0.014011   
15  1  0.5   1   1  1.0  1.00000  1.00000  0.0 -0.02 -26.462  0.003580   
9   1  0.5   1   1  1.0  1.00000  1.00000  0.0 -0.02 -24.175  0.007765   
3   1  0.5   1   1  1.0  1.00000  1.00000  0.0 -0.02 -22.690  0.009050   
16  1  0.5  -1   1  1.0  1.00000  1.00000  0.0 -0.02 -18.724  0.003821   
21  1  0.5  -1   1  1.0  1.00000  1.00000  0.0 -0.02 -14.686  0.016781   
10  1  0.5  -1   1  1.0  1.00000  1.00000  0.0 -0.02 -14.034  0.019014   
4   1  0.5  -1   1  1.0  1.00000  1.00000  0.0 -0.02 -12.598  0.012657   
17  1  0.5   1   1  1.0  0.99999  0.99998  0.0 -0.02 -10.530  0.005566   
26  1  0.5   1   1  1.0  0.99943  0.00001  0.0 -0.02  -5.238  0.027355   
22  1  0.5   1   1  

In [38]:
import pandas as pd
import re
from io import StringIO  # For reading string data into DataFrame

# File path (replace with your actual file path)
file_path = 'out_details.ts1'

# Read the entire file into memory
with open(file_path, 'r') as file:
    content = file.read()

# Use regex to find all relevant tables
# This assumes each table starts with 'Nr' and ends with a blank line
tables = re.findall(r'Nr.*?(?=\n\s*\n)', content, re.DOTALL)

# Initialize a list to hold DataFrames for each table
dataframes = []

# Process each table
for i, table in enumerate(tables):
    # Split the table into lines and clean up
    lines = table.strip().split('\n')
    
    # Check if we have enough lines for headers and data
    if len(lines) > 1:
        # Prepare a list to hold cleaned lines
        cleaned_lines = []

        # Clean each line and ensure consistent spacing
        for line in lines:
            cleaned_line = re.sub(r'\s+', ' ', line.strip())  # Replace multiple spaces with a single space
            cleaned_lines.append(cleaned_line)

        # Join cleaned lines for parsing
        cleaned_table = '\n'.join(cleaned_lines)

        try:
            # Load the relevant data into a pandas DataFrame
            df = pd.read_csv(StringIO(cleaned_table), delim_whitespace=True, header=None)

            # Dynamically assign the first row as headers
            df.columns = df.iloc[0]  # Set the first row as the header
            df = df[1:]  # Remove the header row from the data
            
            # Reset index
            df.reset_index(drop=True, inplace=True)

            # Store the DataFrame
            dataframes.append(df)
        except pd.errors.ParserError as e:
            print(f"ParserError for Table {i + 1}: {e}")
        except pd.errors.EmptyDataError:
            print(f"Empty DataFrame encountered for Table {i + 1}. Skipping...")

# Display extracted DataFrames and find maximum energy for J_z = 0.5 in each
for i, df in enumerate(dataframes):
    print(f"Table {i + 1}:")
    print(df)
    
    # Check if 'j_z' and 'energy' columns exist
    if 'j_z' in df.columns and 'energy' in df.columns:
        # Filter for J_z = 0.5 and find the maximum energy
        filtered_df = df[df['j_z'] == 0.5]
        if not filtered_df.empty:
            max_energy = filtered_df['energy'].max()
            print(f"The maximum energy corresponding to J_z = 0.5 in Table {i + 1} is: {max_energy}\n")
        else:
            print(f"No data for J_z = 0.5 found in Table {i + 1}.\n")
    else:
        print(f"Columns 'j_z' or 'energy' not found in Table {i + 1}.\n")


ParserError for Table 2: Error tokenizing data. C error: Expected 12 fields in line 2, saw 13

ParserError for Table 3: Error tokenizing data. C error: Expected 12 fields in line 2, saw 13

ParserError for Table 4: Error tokenizing data. C error: Expected 12 fields in line 2, saw 13

ParserError for Table 5: Error tokenizing data. C error: Expected 12 fields in line 2, saw 13

Table 1:
0     Nr Iso  j_z total   r  z    v^2  1.0  2.0 osc.quant
0      1   1  0.5     1   1  1  1.000  1.0  0.0     1.501
1      2   1  0.5    -1   1  1  1.000  0.0  1.0     2.606
2      3   1  0.5     1   1  1  1.000  1.0  0.0     3.552
3      4   1  0.5    -1   1  1  1.000  0.0  1.0     4.658
4      5   1  0.5     1   1  1  0.000  1.0  0.0     5.603
..   ...  ..  ...   ...  .. ..    ...  ...  ...       ...
238  239   2  6.5    -1   1  1  0.000  1.0  0.0     8.223
239  240   2  6.5     1   1  1  0.000  1.0  0.0     9.174
240  241   2  7.5     1  -1  1  0.000  1.0  0.0     8.234
241  242   2  7.5    -1  -1  1 

In [3]:
import pandas as pd
from io import StringIO

# File path (replace with your actual file path)
file_path = 'out_details.tst'

# Read the entire file into memory
with open(file_path, 'r') as file:
    content = file.read()

# Split content into potential table blocks based on newlines
# We're looking for blocks that contain 'Nr' and 'energy'
tables = content.split('\n\n')  # Split by double newlines

# Initialize a list to hold DataFrames for each table
dataframes = []

# Process each table block
for i, table in enumerate(tables):
    # Check if the block contains both 'Nr' and 'energy'
    if 'Nr' in table and 'energy' in table:
        print(f"\nProcessing Table {i + 1}:\n{table[:200]}...")  # Print the first 200 characters of each table for debugging
        
        # Prepare a cleaned version of the table
        cleaned_table = '\n'.join(line.strip() for line in table.strip().splitlines())
        
        try:
            # Convert the cleaned table into a DataFrame
            df = pd.read_csv(StringIO(cleaned_table), delim_whitespace=True, header=None)

            # Assign the first row as headers and remove it from data
            df.columns = df.iloc[0]
            df = df[1:]

            # Reset index
            df.reset_index(drop=True, inplace=True)

            # Store the DataFrame
            dataframes.append(df)
            
            # Print the first few rows of the DataFrame to verify it's working
            print(f"\nExtracted DataFrame from Table {i + 1}:")
            print(df.head())
            
        except pd.errors.EmptyDataError:
            print(f"Empty DataFrame encountered for Table {i + 1}. Skipping...")
        except Exception as e:
            print(f"An error occurred for Table {i + 1}: {e}")

# Display extracted DataFrames and find maximum energy for J_z = 0.5 in each
for i, df in enumerate(dataframes):
    print(f"\nFinal Data from Table {i + 1}:")
    print(df)
    
    # Check if 'j_z' and 'energy' columns exist
    if 'j_z' in df.columns and 'energy' in df.columns:
        # Filter for J_z = 0.5 and find the maximum energy
        filtered_df = df[df['j_z'] == 0.5]
        if not filtered_df.empty:
            max_energy = filtered_df['energy'].max()
            print(f"The maximum energy corresponding to J_z = 0.5 in Table {i + 1} is: {max_energy}\n")
        else:
            print(f"No data for J_z = 0.5 found in Table {i + 1}.\n")
    else:
        print(f"Columns 'j_z' or 'energy' not found in Table {i + 1}.\n")


In [13]:
import pandas as pd

# Sample DataFrame
data = {
    'A': [1, 2, 3],
    'B': [4.0, 5.1, 'text'],  # Last row has a non-numeric value
    'C': [7, 8, 9]
}
df = pd.DataFrame(data)

# Check if the last row consists of numbers
last_row = df.iloc[-1]  # Get the last row
is_numeric = pd.to_numeric(last_row, errors='coerce').notnull().all()

print("The last row consists entirely of numbers:", is_numeric)


The last row consists entirely of numbers: False
